# Day 1: getting to know the data

**Python for Data Analysis**, Coding for Reproducible Research (CfRR), University of Exeter

## The question

Was summer 2026 actually unusual, or does it just feel that way? Over two sessions we answer it with the Met Office's monthly weather record for Oxford, which starts in 1853. Today we get the data into shape and take a first look. Tomorrow we plot it, fit a trend and bring in a second station.

## By the end of today you will be able to

- read a messy text file into a pandas DataFrame, choosing the separator, header and missing-value markers yourself
- explain how one stray character turns a column of numbers into text, and fix it without throwing information away
- say what a NumPy array is, why vectorised code beats a loop, and how NaN behaves in NumPy and pandas
- decide what to do about missing data, and defend the decision
- summarise data with split-apply-combine (`groupby`)

## How to use this notebook

There are two versions of this notebook. In `day1.ipynb` the explanations are written for you and most code cells are empty: in the session, you type the code into them as the instructor types it on screen. A few cells, mostly setup, are filled in already; just run those. [`day1_solutions.ipynb`](day1_solutions.ipynb), in the same folder, has every cell filled in, with its output. If you fall behind, copy the cell you missed from there and carry on. Afterwards, either one is your notes.

The [CfRR Python for Data Analysis pages](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/section_landing_pages/python_for_data_analysis.html) are the full reference. This notebook links to them when you might want every option.

*Data: Met Office historic station data for Oxford. Contains public sector information licensed under the [Open Government Licence v3.0](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).*

## 1. Welcome, the question and the data

*10:00, 15 minutes*

Before any code: was summer 2026 in Oxford unusual?

### Meet the data

The Met Office publishes [monthly records](https://www.metoffice.gov.uk/research/climate/maps-and-data/historic-station-data) for a set of long-running UK weather stations: one plain text file per station, free to reuse. We will use Oxford, whose record starts in January 1853. The station is at 51.76°N, 1.26°W, 63 metres above sea level.

Each month in the file is summarised by five numbers:

| Column | What it measures | Units |
|---|---|---|
| `tmax` | Mean daily maximum temperature: each day's highest temperature, averaged over the month | °C |
| `tmin` | Mean daily minimum temperature: each night's lowest temperature, averaged over the month | °C |
| `af` | Days of air frost: days when the air temperature fell below 0 °C | days |
| `rain` | Total rainfall for the month | mm |
| `sun` | Total hours of sunshine in the month | hours |

Two things are easy to misread:

- **`tmax` is an average, not a record.** A July `tmax` of 22 °C means the daily highs averaged 22 °C. Some days were much hotter.
- **"Summer" means June, July and August.** That is the meteorologists' definition, and the one we will use.

The record is not perfect (data very rarely is!):

- **Estimated** values fill gaps using the Met Office's gridded dataset (HadUK-Grid) rather than a reading at the station.
- **Missing** values are months with more than two days of data missing.
- **Provisional** values have not yet been through the Met Office's full quality control, and may be revised.
- **Sunshine** was measured for most of the record by a Campbell-Stokes recorder, a glass ball that focuses the sun to burn a trace on a card. Some stations now use an automatic sensor instead.
- Over 170 years, the site and instruments may have changed slightly.

Knowing what each number means, and how it was made, is part of the analysis: it decides which questions the data can answer.

## 2. First look at the raw file

*10:15, 15 minutes*

A notebook runs cells in the order you run them, not the order they appear on the page. Run cells from the top with Shift+Enter. If things get confusing, use **Kernel > Restart Kernel and Run All Cells**.

### The tools

- **pandas** is the standard Python library for tables of data. Its central object is the **DataFrame**: rows and named columns, like a spreadsheet that you drive with code. Every step is written down, so anyone can rerun it and get the same result, which is hard to promise for clicks in a spreadsheet.
- **NumPy** does the fast arithmetic underneath pandas. We look at it properly later this morning.
- **pathlib** builds file locations that work on Windows, macOS and Linux alike.

`import pandas as pd` and `import numpy as np` are conventions: almost all the pandas code you will find elsewhere uses the same short names.

The data came in the zip with this notebook, in the `data` folder next to it. It is a dated snapshot of the Met Office file rather than the live one, so everyone works with exactly the same numbers, and they stay the same when the Met Office updates its own file. This first cell is filled in for you: run it. It should show `True`, meaning Python found the file.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
# Snapshot of https://www.metoffice.gov.uk/pub/data/weather/uk/climate/stationdata/oxforddata.txt
OXFORD = DATA / "oxforddata.txt"
OXFORD.exists()

True

Look at a data file before you load it. In the file browser on the left, open the `data` folder and double-click `oxforddata.txt`. JupyterLab opens it as plain text in a new tab. Use JupyterLab rather than Excel, which would try to split it into columns for you.

Read the top, then jump to the bottom with Ctrl+End (Cmd+Down on a Mac). What will make this file awkward to load into a table?

Six things about this file will make it awkward to load. How many did you find?

- five lines of free text before the column names
- a row of units under the column names
- columns separated by spaces rather than commas
- `---` for missing values
- a `*` after estimated values
- the word `Provisional` at the end of recent rows

Each one causes a problem in the next half hour. The obvious first try is `pd.read_csv` with no options at all:

In [2]:
naive = pd.read_csv(OXFORD)
naive.shape

(2091, 1)

In [3]:
naive.head()

,,Oxford
Location: 450900E 207200N,Lat 51.761 Lon -1.262,63 metres amsl
Estimated data is marked with a * after the value.,nan,NaN
Missing data (more than 2 days missing in month) is marked by ---.,nan,NaN
Sunshine data taken from an automatic Kipp & Zonen sensor marked with a #,otherwise sunshine data taken from a Campbell Stokes recorder.,NaN
yyyy mm tmax tmin af rain sun,nan,NaN


One column. `read_csv` splits on commas by default and there are none, so each whole line becomes a single value, and the first line (`Oxford`) becomes the column name.

## 3. Loading it properly

*10:30, 30 minutes*

"CSV" means comma-separated values, but `read_csv` reads any delimited text once you tell it how. Reading a file is a set of decisions, not a default:

- **`sep`**: what separates the columns
- **`skiprows`**: how many lines to skip before the data
- **`names`**: what to call the columns
- **`na_values`**: how this file writes "missing"

The [CfRR Pandas page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_Pandas.html) covers the other options.

First the separator. `sep=r"\s+"` means "one or more whitespace characters".

In [4]:
pd.read_csv(OXFORD, sep=r"\s+")

ParserError: Error tokenizing data. C error: Expected 10 fields in line 4, saw 13


An error. Read error messages from the bottom up: the last line says what went wrong, and everything above it says where. Here pandas expected 10 fields on line 4 and found 13. Line 4 is free text, not data.

So we skip all seven lines before the first row of numbers, and supply the column names ourselves.

In [5]:
COLUMNS = ["yyyy", "mm", "tmax", "tmin", "af", "rain", "sun"]
pd.read_csv(OXFORD, sep=r"\s+", skiprows=7, names=COLUMNS)

ParserError: Error tokenizing data. C error: Expected 7 fields in line 2084, saw 8


A new error, about line 2084. To see that line, read the file into a list with one string per line. Python counts from 0, so line 2084 is `lines[2083]`.

In [6]:
lines = OXFORD.read_text().splitlines()
print(lines[2083])

   2026   1    7.9*    2.1*     11*   34.1    71.3*  Provisional


Recent rows have an eighth field, the word `Provisional`. The data are ragged: some rows are longer than others. That word tells us these numbers may still change, so rather than throw it away we keep it as a column of its own, `status`. While we are at it, `na_values` tells pandas how this file writes "missing".

In [7]:
COLUMNS = ["yyyy", "mm", "tmax", "tmin", "af", "rain", "sun", "status"]
df = pd.read_csv(OXFORD, sep=r"\s+", skiprows=7, names=COLUMNS, na_values="---")
df.head()

,yyyy,mm,tmax,tmin,af,rain,sun,status
0,1853,1,8.4,2.7,4,62.8,NaN,NaN
1,1853,2,3.2,-1.8,19,29.3,NaN,NaN
2,1853,3,7.7,-0.6,20,25.9,NaN,NaN
3,1853,4,12.6,4.5,0,60.1,NaN,NaN
4,1853,5,16.8,6.1,0,59.5,NaN,NaN


In [8]:
df.tail()

,yyyy,mm,tmax,tmin,af,rain,sun,status
2080,2026,5,20.7*,9.6*,0*,12.0,239.1*,Provisional
2081,2026,6,23.8*,14.4*,0*,58.7,231.9*,Provisional
2082,2026,7,28.2*,14.9*,0*,0.0,353.0*,Provisional
2083,2026,8,26.5*,14.3*,0*,28.4,267.9*,Provisional
2084,2026,9,22.2*,12.9*,0*,28.1*,149.1*,Provisional


### Anatomy of a DataFrame

Now the data are in, it is worth knowing the parts:

- the **columns**, each with a name
- the **index**: the row labels down the left (for now just 0, 1, 2 and so on)
- each column on its own is a **Series**: one column of values that keeps the index

`shape` gives the number of rows and columns. Selecting one column gives a Series: look at the last line of its output, `dtype`. We will need it soon.

In [9]:
df.shape

(2085, 8)

In [10]:
df["tmax"].head(3)

0    8.4
1    3.2
2    7.7
Name: tmax, dtype: str

`na_values="---"` turned the file's own missing marker into proper missing values (`NaN`). Count them in each column:

In [11]:
df.isna().sum()

yyyy         0
mm           0
tmax         0
tmin         1
af           1
rain         0
sun        912
status    2076
dtype: int64

`sun` has the most: sunshine was not recorded at Oxford until 1929. `status` is mostly missing too, but that is expected, because only provisional rows have a status. One month, December 1860, is missing `tmin` and `af`.

## 4. The asterisk problem

*11:00, 30 minutes*

The first thing anyone wants from a column of temperatures is its average.

In [12]:
df["tmax"].mean()

TypeError: Cannot perform reduction 'mean' with string dtype

It fails. Check the data type of each column:

In [13]:
df.dtypes

yyyy      int64
mm        int64
tmax        str
tmin        str
af          str
rain        str
sun         str
status      str
dtype: object

`tmax` is stored as text (`str` in pandas 3, `object` in earlier versions). One value pandas cannot read as a number is enough to make the whole column text. Here are the culprits:

In [14]:
df[df["tmax"].str.endswith("*")].tail()

,yyyy,mm,tmax,tmin,af,rain,sun,status
2080,2026,5,20.7*,9.6*,0*,12.0,239.1*,Provisional
2081,2026,6,23.8*,14.4*,0*,58.7,231.9*,Provisional
2082,2026,7,28.2*,14.9*,0*,0.0,353.0*,Provisional
2083,2026,8,26.5*,14.3*,0*,28.4,267.9*,Provisional
2084,2026,9,22.2*,12.9*,0*,28.1*,149.1*,Provisional


The `*` marks an estimated value. That matters: July 2026, the month we care about, is one of them.

### The quick fix, and why it is wrong

Ask an AI assistant, or search online, and you will often be told to use `pd.to_numeric(df["tmax"], errors="coerce")`. Try it, and count the missing values it leaves:

In [15]:
suggested = pd.to_numeric(df["tmax"], errors="coerce")
suggested.isna().sum()

np.int64(26)

`errors="coerce"` turns anything it cannot read into `NaN`, without telling you. Every estimated value disappears, including July 2026. The code runs and the answer is wrong. That is the worst kind of bug.

A better approach keeps the information: record which values were estimated in a column of its own, then strip the `*` and convert. `.astype(float)` fails loudly if it meets anything else it does not expect, which is what we want.

In [16]:
# Keep the flag, then strip the marker and convert
estimated = df["tmax"].str.endswith("*")
tmax_clean = df["tmax"].str.rstrip("*").astype(float)
estimated.sum(), tmax_clean.mean()

(np.int64(26), np.float64(14.029112709832134))

All five measured columns need the same treatment, and the results should go back into `df`. A loop does it once for all of them: it replaces each column with its cleaned version and adds an `_estimated` column alongside it.

`astype(str)` first makes sure every column is text before we look for a `*`. At some stations a column has no estimates at all, so pandas reads it as numbers already, and `.str` only works on text.

In [17]:
MEASURES = ["tmax", "tmin", "af", "rain", "sun"]
for col in MEASURES:
    text = df[col].astype(str)
    df[col + "_estimated"] = text.str.endswith("*")
    df[col] = text.str.rstrip("*").astype(float)
df.dtypes

yyyy                int64
mm                  int64
tmax              float64
tmin              float64
af                float64
rain              float64
sun               float64
status                str
tmax_estimated       bool
tmin_estimated       bool
af_estimated         bool
rain_estimated       bool
sun_estimated        bool
dtype: object

The `status` column gets the same idea: a `True`/`False` column called `provisional` keeps the information, and then `status` can go.

In [18]:
df["provisional"] = df["status"] == "Provisional"
df = df.drop(columns="status")
df[["yyyy", "mm", "tmax", "tmax_estimated", "provisional"]].tail(3)

,yyyy,mm,tmax,tmax_estimated,provisional
2082,2026,7,28.2,True,True
2083,2026,8,26.5,True,True
2084,2026,9,22.2,True,True


We have kept every flag as its own column rather than deleting the estimated and provisional rows. Throwing them away would have thrown away July 2026.

## Break

*11:30, 15 minutes*

## 5. What's underneath: NumPy

*11:45, 35 minutes*

Every pandas column is a NumPy array with a name and an index attached. NumPy is what makes the arithmetic fast. The [CfRR NumPy page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_NumPy.html) has the full tour.

In [19]:
tmax = df["tmax"].to_numpy()
tmin = df["tmin"].to_numpy()
type(tmax), tmax.dtype, tmax.shape

(numpy.ndarray, dtype('float64'), (2085,))

The mean temperature for a month is the average of its mean maximum and mean minimum. Here it is the way the Introduction to Python course would write it, with a loop:

In [20]:
tmean_loop = []
for i in range(len(tmax)):
    tmean_loop.append((tmax[i] + tmin[i]) / 2)
tmean_loop[:3]

[np.float64(5.550000000000001),
 np.float64(0.7000000000000001),
 np.float64(3.5500000000000003)]

NumPy can do the same calculation without a loop. Arithmetic on an array applies to every element at once: this is called **vectorisation**.

In [21]:
tmean = (tmax + tmin) / 2
tmean[:3]

array([5.55, 0.7 , 3.55])

Same numbers, less code. It is also much faster. `%timeit` runs a line many times and reports how long it takes:

In [22]:
%timeit [(tmax[i] + tmin[i]) / 2 for i in range(len(tmax))]

322 μs ± 4.97 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [23]:
%timeit (tmax + tmin) / 2

1.7 μs ± 7.01 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


The vectorised version is many times faster, and the gap grows with the size of the data. That is why pandas and NumPy code avoids looping over rows.

### How NaN behaves

December 1860 has no `tmin`. Here are three ways to average `tmin`, missing month included:

In [24]:
np.mean(tmin)

np.float64(nan)

In [25]:
np.nanmean(tmin)

np.float64(6.280278310940498)

In [26]:
df["tmin"].mean()

np.float64(6.280278310940498)

`np.mean` returns `nan`: any arithmetic with `NaN` gives `NaN`, so one missing month spoils the whole result. `np.nanmean` skips missing values. pandas skips them by default. Neither choice is wrong, but you need to know which one you are getting.

Add the monthly mean temperature to the DataFrame. pandas lines the values up by row for us.

In [27]:
df["tmean"] = (df["tmax"] + df["tmin"]) / 2

### Boolean masks

A comparison on a column gives a column of `True`/`False` values, which can pick out rows.

In [28]:
frost_free = df["af"] == 0
frost_free.sum()

np.int64(1009)

Use the mask to pick out the frost-free rows and their `mm` column. `.value_counts()` then counts how often each month of the year appears: it is the quickest way to see what is in a column.

In [29]:
df.loc[frost_free, "mm"].value_counts()

mm
6     174
7     174
8     174
9     173
5     132
10     93
4      43
11     18
12      8
2       8
3       8
1       4
Name: count, dtype: int64

Every June, July and August in the record is frost-free, as you would hope.

### Missing data is a decision

`df.dropna()` removes every row with a missing value. How many rows would survive?

In [30]:
len(df), len(df.dropna())

(2085, 1173)

Only just over half. Sunshine is missing for every month from 1853 to 1928, so dropping those rows also throws away 76 years of perfectly good temperature records.

There are four things we could do about the missing sunshine, and each has a cost:

1. **Drop every row with a missing value.** Loses most of the temperature record from before 1929.
2. **Drop the `sun` column.** Loses the sunshine we do have.
3. **Fill the gaps with the average sunshine.** Invents data, as the next cell shows.
4. **Keep everything, leave the gaps as `NaN`, and only use sunshine from 1929.**

We take option 4. pandas already skips `NaN` in calculations, so the gaps do no harm as long as we know they are there. Option 3 is the most dangerous, because it looks harmless:

In [31]:
filled = df["sun"].fillna(df["sun"].mean())
df["sun"].std(), filled.std()

(np.float64(63.86273077602531), np.float64(47.89194042558409))

Filling invents 76 years of perfectly average sunshine, and makes the record look far less variable than it was. Any analysis of sunshine over time would be quietly wrong.

The single missing `tmin` can stay as `NaN` too. Filling one month in a long record would also be defensible, as long as you say so.

## 6. Summarising: split-apply-combine

*12:20, 30 minutes*

First, give each row a proper date. A datetime index lets pandas understand months and years. This cell is filled in: the syntax is fiddly and not worth memorising.

In [32]:
df.index = pd.to_datetime({"year": df["yyyy"], "month": df["mm"], "day": 1})
df[["yyyy", "mm", "tmax", "tmean"]].head(3)

,yyyy,mm,tmax,tmean
1853-01-01,1853,1,8.4,5.55
1853-02-01,1853,2,3.2,0.70
1853-03-01,1853,3,7.7,3.55


Before writing code, say the plan in plain English. To get the average summer (June to August) maximum temperature for each year:

1. keep only the June, July and August rows
2. **split** them into groups by year
3. **apply** the mean to `tmax` in each group
4. **combine** the results into one table

That is split-apply-combine, and `groupby` does the last three steps in one line.

In [33]:
summer = df[df["mm"].isin([6, 7, 8])]
summer_tmax = summer.groupby("yyyy")["tmax"].mean()
summer_tmax.tail()

yyyy
2022    24.666667
2023    22.266667
2024    21.966667
2025    24.500000
2026    26.166667
Name: tmax, dtype: float64

Group by month instead of year, using all the rows, and the same line answers a different question:

In [34]:
df.groupby("mm")["tmax"].mean()

mm
1      6.827011
2      7.586782
3     10.123563
4     13.330460
5     16.853448
6     20.054598
7     21.959770
8     21.400575
9     18.658621
10    14.270520
11     9.778613
12     7.444509
Name: tmax, dtype: float64

This is the average maximum temperature for each month of the year over the whole record: the seasonal cycle.

Now rank every July in the record by `tmax`, hottest first, and show the top five with their flags:

In [35]:
julys = df[df["mm"] == 7].sort_values("tmax", ascending=False)
julys[["tmax", "tmax_estimated", "provisional"]].head()

,tmax,tmax_estimated,provisional
2026-07-01,28.2,True,True
2018-07-01,27.4,False,False
2006-07-01,27.1,False,False
1983-07-01,26.8,False,False
1911-07-01,26.1,False,False


## 7. Cliffhanger

*12:50, 10 minutes*

Where does summer 2026 come in the ranking of whole summers?

In [36]:
summer_tmax.sort_values(ascending=False).head()

yyyy
2026    26.166667
1976    24.900000
2022    24.666667
2018    24.533333
2025    24.500000
Name: tmax, dtype: float64

So July 2026 is the hottest July on record, and summer 2026 the hottest summer. Is that enough to say it was unusual?

- The 2026 temperatures are estimated and provisional.
- In any long record, some year has to be the hottest. Is 2026 far ahead, or just a bit ahead?
- Summers have been getting warmer. Is 2026 unusual for its time, or just the latest step on a trend?

Tomorrow we plot it, fit a trend and find out.

### Today's ideas

- Reading a file is a set of decisions: `sep`, `skiprows`, `names`, `na_values`.
- One stray character turns a numeric column into text. Fix it and keep the information as a flag.
- `errors="coerce"` hides problems. Prefer code that fails loudly.
- A column is a NumPy array. Vectorised arithmetic beats loops.
- `NaN` spreads in NumPy and is skipped in pandas.
- Dropping or filling missing data is a decision with consequences.
- `groupby` is split-apply-combine.